# Qwen3-1.7B Foundation LLM
Fine-tuned on `ChatDoctor_HealthCareMagic_112k` using LoRA.

Preprocessing pipeline was created by Aaron.

In [ ]:
!pip install torch transformers accelerate peft datasets trl bitsandbytes sacremoses
!pip install --upgrade "torchao>=0.16.0"
!pip install contractions textblob
!python -m textblob.download_corpora lite

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
Finished.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import re
import contractions
from textblob import TextBlob
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset, DatasetDict

assert torch.cuda.is_available(), "CUDA not available. Please enable GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU: Tesla T4
VRAM: 14.6 GB


## Preprocessing Pipeline

- Clean punctuation and regex noise while preserving clinical abbreviations and decimals.
- Expand contractions and protect medical terms during typo correction.
- Normalize text with ClinicalBERT, then run the full pipeline in order.

In [ ]:
def clean_punctuation(text):
    
    # Clean punctuation while preserving clinical abbreviations and decimals.
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Preserve clinical abbreviations and decimal numbers.
    text = re.sub(r'i\.e\.', '_LOCK_IE_', text)
    text = re.sub(r'\bi\.e\b', '_LOCK_IE_', text)
    text = re.sub(r'e\.g\.', '_LOCK_EG_', text)
    text = re.sub(r'\be\.g\b', '_LOCK_EG_', text)
    text = re.sub(r'dr\.', '_LOCK_DR_', text)

    text = re.sub(r'[\(\)]', ' ', text)
    text = re.sub(r'([a-z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-z])', r'\1 \2', text)
    text = text.replace('?', ' ')
    text = re.sub(r'(\d+)\.(\d+)', r'\1_SAFE_DEC_\2', text)
    text = re.sub(r'\.{2,}', ' ', text)
    text = re.sub(r'([a-z0-9])\.([a-z])', r'\1 \2', text)
    text = re.sub(r'[.,;!]+(?=\s|$)', '', text)
    text = re.sub(r'\s\.\s', ' ', text)

    text = text.replace('_LOCK_IE_', 'i.e')
    text = text.replace('_LOCK_EG_', 'e.g')
    text = text.replace('_LOCK_DR_', 'dr.')
    text = text.replace('_SAFE_DEC_', '.')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def expand_contractions(text):
   
    # Expand English contractions with the contractions library.
    if not isinstance(text, str):
        return ""
    text = contractions.fix(text)
    return re.sub(r'\s+', ' ', text).strip()


print("Loading ClinicalBERT tokenizer for normalization...")
clinical_tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")

# Preserve medical terms that should not be autocorrected.
MEDICAL_SAFETY_SHIELD = {
    'panadol', 'nurofen', 'pooing', 'weezy', 'croupy', 'raspy',
    'rattly', 'uti', 'ild', 'hpylori', 'gyno', 'cause'
}

def autocorrect_typos(text):
    
    # Fix known typos while skipping medical safety terms.
    words = text.split()
    corrected = []
    for word in words:
        clean_word = re.sub(r'[^a-z0-9]', '', word.lower())
        if clean_word in MEDICAL_SAFETY_SHIELD:
            corrected.append(word)
        elif clean_word in ("immediatly", "afect", "experiencing"):
            corrected.append(str(TextBlob(word).correct().lower()))
        else:
            corrected.append(word)
    return " ".join(corrected)

def normalize_with_clinicalbert(text):
    
    # Normalize medical text with ClinicalBERT tokenization.
    if not isinstance(text, str):
        return ""
    text = expand_contractions(text)
    text = autocorrect_typos(text)
    tokens = clinical_tokenizer.tokenize(text)
    clean_text = clinical_tokenizer.convert_tokens_to_string(tokens)
    clean_text = clean_text.replace(" .", ".").replace(" ,", ",").strip()
    return clean_text


def full_preprocess(text):
    
    text = clean_punctuation(text)
    text = normalize_with_clinicalbert(text)
    return text

print("Preprocessing pipeline ready.")

sample = "I've been having chest pain...it's really bad!!! Dr. Smith said my temp is 98.6"
print(f"\nOriginal : {sample}")
print(f"Processed: {full_preprocess(sample)}")

Loading ClinicalBERT tokenizer for normalization...
Preprocessing pipeline ready.

Original : I've been having chest pain...it's really bad!!! Dr. Smith said my temp is 98.6
Processed: i have been having chest pain it is really bad dr. smith said my temp is 98. 6


## Model Loading & LoRA Configuration

In [ ]:
model_name = "Qwen/Qwen3-1.7B"

qwen_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

# Qwen3-1.7B runs in FP16 on the target GPU.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"":0},
    trust_remote_code=True
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Qwen uses o_proj
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 6,422,528 || all params: 2,038,162,432 || trainable%: 0.3151


## Load Dataset & Apply Preprocessing

In [ ]:
raw_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
raw_dataset = raw_dataset.select(range(5000))
print(f"Total samples: {len(raw_dataset)}")

system_prompt = (
    "You are an experienced family doctor giving clear, direct medical advice. "
    "Answer in second person. Do not roleplay as a patient."
)

# Compile patterns once for efficiency.
OPEN_PAT = re.compile(
    r'^(I have gone through[^.]*\.|Thanks for your[^.]*\.|I am Chat[^.]*\.|'
    r'I gone through[^.]*\.|Welcome to[^.]*\.|Hello[^.]*\.|Hi[^.]*\.|'
    r'I can understand[^.]*\.|I understand your[^.]*\.)\s*',
    flags=re.IGNORECASE
)
TRAIL_PAT = re.compile(
    r'\s*(Hope\b.*|Wish\b.*|Take care\b.*|Kindly\b.*|Please (?:consult|advise|help)\b.*|'
    r'Let me know\b.*|Thanks for using\b.*|Chat\s?[Dd]octor\b.*)',
    flags=re.IGNORECASE | re.DOTALL
)

def format_chat(example):
    convs = example["conversations"]
    if len(convs) != 2 or convs[0]["from"] != "human" or convs[1]["from"] != "gpt":
        return {"text": ""}

    user_msg      = convs[0]["value"].strip()
    assistant_msg = convs[1]["value"].strip()

    # Apply preprocessing to the user message.
    user_msg = full_preprocess(user_msg)

    # Clean assistant response.
    assistant_clean = OPEN_PAT.sub("", assistant_msg).strip()
    assistant_clean = TRAIL_PAT.sub("", assistant_clean).strip()
    assistant_clean = re.sub(r'\bwe\b', 'you', assistant_clean, flags=re.IGNORECASE)
    assistant_clean = re.sub(r'\bour\b', 'your', assistant_clean, flags=re.IGNORECASE)
    assistant_clean = assistant_clean[:400]

    # Drop samples where cleaning removed all content.
    if len(assistant_clean) < 30:
        return {"text": ""}

    text = f"{system_prompt}\nUser: {user_msg}\nAssistant: {assistant_clean}\n"
    return {"text": text}

print("Applying preprocessing pipeline to dataset (this may take a few minutes)...")
dataset = raw_dataset.map(format_chat, remove_columns=raw_dataset.column_names)
dataset = dataset.filter(lambda x: x["text"] != "")
print(f"After preprocessing & cleaning: {len(dataset)} samples")

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

Total samples: 5000
Applying preprocessing pipeline to dataset (this may take a few minutes)...
After preprocessing & cleaning: 4600 samples
Train: 4140 | Eval: 460


In [ ]:
# Show 3 samples to verify preprocessing is working correctly
print("Sample preprocessed training examples:\n")
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print(train_dataset[i]["text"][:400])
    print()

Sample preprocessed training examples:

--- Sample 1 ---
You are an experienced family doctor giving clear, direct medical advice. Answer in second person. Do not roleplay as a patient.
User: my dad is 74 his kidneys are functioning at 6 % he has enlarged prostrate glands but the doctors are not concerned of this at the moment he is not on dialysis and is doing quite well for a person who is kidney function is at 6 % can prostrate glands affect the kidn

--- Sample 2 ---
You are an experienced family doctor giving clear, direct medical advice. Answer in second person. Do not roleplay as a patient.
User: i am 40 yrs old female wt = 65 kg ht = 5 ft 2 in i have two daughters my husband is also a teacher i teach in high school my problem is knee pain and some times pain in middle fingers quite often and pain in my leg below the knees i feel that i am lossing my memory

--- Sample 3 ---
You are an experienced family doctor giving clear, direct medical advice. Answer in second person. Do not

In [ ]:
def tokenize_fn(examples):
    tokenized = qwen_tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval  = eval_dataset.map(tokenize_fn,  batched=True, remove_columns=["text"])

tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format(type="torch",  columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done. Example shape:", tokenized_train[0]["input_ids"].shape)

Tokenization done. Example shape: torch.Size([512])


In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen3-1.7b-familydoctor", 
    per_device_train_batch_size=1,  # Keep GPU memory usage low
    gradient_accumulation_steps=8, 
    num_train_epochs=3,
    learning_rate=2e-4,  
    fp16=True, 
    logging_steps=50,
    save_steps=400,  # Periodic checkpoint saving
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,  # Restore best checkpoint after training
    metric_for_best_model="eval_loss",
    warmup_steps=100,
    report_to="none",
    gradient_checkpointing=True,  # Reduce memory during backprop
    optim="adafactor",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=qwen_tokenizer,
    mlm=False,  # Causal LM training
    pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Stop if eval loss stops improving
)

trainer.train()

model.save_pretrained("./qwen3-1.7b-familydoctor-final")
qwen_tokenizer.save_pretrained("./qwen3-1.7b-familydoctor-final")
print("Training complete and model saved.")

Step,Training Loss,Validation Loss
200,2.539213,2.541183
400,2.516711,2.503972
600,2.452015,2.490021
800,2.442477,2.474667
1000,2.411075,2.465989
1200,2.376860,2.464917
1400,2.366291,2.462095


Training complete and model saved.
